# 05 — Fusion Model Training

Loads pretrained encoders (from notebook 04), builds the LateFusionTransformer, trains with CombinedLoss (FocalLoss + DQ SmoothL1), encoder freeze for first 10 epochs, modality dropout strategy, early stopping by val AUC.

**Runtime: ~2–3 hours on T4 GPU.**

**Run order: 01 → 02 → 03 → 04 → 05 → 06**

In [ ]:
# CELL 2: Mount Google Drive (checkpoints will be saved here after training)
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/EarlyMind/checkpoints', exist_ok=True)
print('Drive mounted. Fresh .pt files will be saved to MyDrive/EarlyMind/checkpoints/')

In [ ]:
# CELL 1: Clone / pull repo
import os
if not os.path.exists('/content/earlyMind'):
    !git clone https://github.com/Rickykapoor/earlyMind.git /content/earlyMind
%cd /content/earlyMind
!git pull origin main

In [ ]:
# CELL 2: Install dependencies
!pip install -q mne nibabel nilearn openneuro-py torch torchvision \
  streamlit plotly scikit-learn pytorch-tabnet \
  xgboost catboost tqdm pyyaml scipy

In [ ]:
# CELL 3: Check GPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# CELL 4: Download pretrained encoder checkpoints from GitHub Releases
# These are produced by notebook 04. Download them if not already present.
import os

RELEASE = 'https://github.com/Rickykapoor/earlyMind/releases/download/v1.0.0-data'

os.makedirs('checkpoints', exist_ok=True)

for ckpt in ['eeg_encoder_pretrained.pt', 'mri_encoder_pretrained.pt', 'hpo_encoder_pretrained.pt']:
    path = f'checkpoints/{ckpt}'
    if not os.path.exists(path):
        print(f'Downloading {ckpt} ...')
        !wget -qO {path} {RELEASE}/{ckpt}
    else:
        print(f'{ckpt} already present. Skipping.')

# Also download processed datasets if missing
for name, tar, dest in [
    ('eeg', 'eeg_raw.tar.gz',     'datasets/eeg'),
    ('mri', 'mri_raw.tar.gz',     'datasets/mri'),
    ('hpo', 'facial_raw.tar.gz',  'datasets/facial'),
]:
    if not os.path.exists(f'datasets/processed/{name}') and not os.path.exists(f'datasets/{name}'):
        print(f'Downloading {tar} ...')
        !wget -qO {tar} {RELEASE}/{tar}
        !tar -xzf {tar} -C datasets/{name if name != "hpo" else "facial"}

print('All assets ready.')

In [ ]:
import sys, os; sys.path.insert(0, '/content/earlyMind') if '/content/earlyMind' not in sys.path else None; os.chdir('/content/earlyMind') if os.getcwd() != '/content/earlyMind' else None
# CELL 5: Setup
import sys
sys.path.insert(0, '/content/earlyMind')

import torch
import numpy as np
from src.config import cfg

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cfg.paths.makedirs()

# Verify pretrained checkpoints exist
ckpt_dir = cfg.paths.checkpoints
for cname in ['eeg_encoder_pretrained.pt', 'mri_encoder_pretrained.pt', 'hpo_encoder_pretrained.pt']:
    p = ckpt_dir / cname
    print(f'{cname}: {"EXISTS" if p.exists() else "MISSING -- run notebook 04 first!"}')

In [ ]:
import sys, os; sys.path.insert(0, '/content/earlyMind') if '/content/earlyMind' not in sys.path else None; os.chdir('/content/earlyMind') if os.getcwd() != '/content/earlyMind' else None
# CELL 6: Build DataLoaders
from src.data.fusion_dataset import build_dataloaders

train_ldr, val_ldr, test_ldr = build_dataloaders(
    eeg_processed_dir  = cfg.paths.eeg_processed,
    mri_processed_dir  = cfg.paths.mri_processed,
    hpo_processed_dir  = cfg.paths.hpo_processed,
    eeg_raw_dir        = cfg.paths.eeg_raw,
    batch_size         = cfg.training.batch_size,
    modality_dropout_p = 0.3,
    seed               = cfg.training.seed,
)

print(f'Train batches: {len(train_ldr)}')
print(f'Val batches:   {len(val_ldr)}')
print(f'Test batches:  {len(test_ldr)}')

sample = next(iter(train_ldr))
print('\nSample batch keys:', list(sample.keys()))
for k, v in sample.items():
    if isinstance(v, torch.Tensor):
        print(f'  {k}: {v.shape}')

In [ ]:
import sys, os; sys.path.insert(0, '/content/earlyMind') if '/content/earlyMind' not in sys.path else None; os.chdir('/content/earlyMind') if os.getcwd() != '/content/earlyMind' else None
# CELL 7: Build Fusion Model
from src.models.fusion_model import build_fusion_model

n_hpo = cfg.model.hpo_n_features
eeg_ckpt = str(cfg.paths.checkpoints / 'eeg_encoder_pretrained.pt')
mri_ckpt = str(cfg.paths.checkpoints / 'mri_encoder_pretrained.pt')
hpo_ckpt = str(cfg.paths.checkpoints / 'hpo_encoder_pretrained.pt')

model = build_fusion_model(
    n_hpo=n_hpo,
    freeze_encoders=True,
    eeg_ckpt=eeg_ckpt if (cfg.paths.checkpoints / 'eeg_encoder_pretrained.pt').exists() else None,
    mri_ckpt=mri_ckpt if (cfg.paths.checkpoints / 'mri_encoder_pretrained.pt').exists() else None,
    hpo_ckpt=hpo_ckpt if (cfg.paths.checkpoints / 'hpo_encoder_pretrained.pt').exists() else None,
)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {total_params:,}')
print(f'Trainable params: {trainable:,}')

In [ ]:
import sys, os; sys.path.insert(0, '/content/earlyMind') if '/content/earlyMind' not in sys.path else None; os.chdir('/content/earlyMind') if os.getcwd() != '/content/earlyMind' else None
# CELL 8: Train!
from src.training.trainer import train_fusion_model

results = train_fusion_model(
    model=model,
    train_loader=train_ldr,
    val_loader=val_ldr,
    device=device,
    save_dir=str(cfg.paths.checkpoints),
    n_epochs=cfg.training.epochs,
    lr=cfg.training.lr,
    weight_decay=cfg.training.weight_decay,
    patience=cfg.training.patience,
    focal_alpha=cfg.training.focal_alpha,
    focal_gamma=cfg.training.focal_gamma,
    severity_weight=cfg.training.severity_loss_weight,
    grad_clip=cfg.training.grad_clip,
    freeze_encoders_epochs=cfg.training.freeze_encoders_epochs,
)

print(f'Best Val AUC: {results["best_auc"]:.4f}')

In [ ]:
# CELL 9: Plot training curves
import json
import matplotlib.pyplot as plt

with open(str(cfg.paths.reports / 'training_history.json')) as f:
    hist = json.load(f)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist['train_loss'], label='train')
axes[0].plot(hist['val_loss'],   label='val')
axes[0].set_title('Fusion Loss')
axes[0].legend()
axes[1].plot(hist['train_auc'],  label='train')
axes[1].plot(hist['val_auc'],    label='val')
axes[1].axhline(0.85, color='r', linestyle='--', label='AUC target')
axes[1].set_title('Fusion AUC')
axes[1].legend()
plt.tight_layout()
plt.savefig(str(cfg.paths.reports) + '/fusion_training_curves.png', dpi=100)
plt.show()

In [ ]:
# LAST CELL: Save fresh checkpoints to Google Drive
# These .pt files are too large for GitHub — save to Drive, then download
# to your Mac and upload to GitHub Releases (v1.0.0-data).
import shutil, os
drive_dir = '/content/drive/MyDrive/EarlyMind/checkpoints'
os.makedirs(drive_dir, exist_ok=True)

saved = []
for ckpt in ['fusion_model.pt', 'eeg_encoder_pretrained.pt', 'mri_encoder_pretrained.pt', 'hpo_encoder_pretrained.pt']:
    src = f'checkpoints/{ckpt}'
    dst = f'{drive_dir}/{ckpt}'
    if os.path.exists(src):
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / 1e6
        print(f'  Saved {ckpt} ({size_mb:.0f} MB) → Google Drive')
        saved.append(ckpt)
    else:
        print(f'  MISSING: {ckpt}')

print(f'\n{len(saved)} checkpoint(s) saved to Google Drive.')
print('ALL checkpoints saved to Drive. Run notebook 06 to evaluate, then upload .pt files to GitHub Releases.')
print('Next step: Download from Drive → Mac → Upload to GitHub Releases v1.0.0-data')
